# Challenge 1: datos y análisis exploratorio

Tu primera responsabilidad es preparar una fuente de datos confiable para el proyecto de clasificación de vinos.

**Entrada:** `data/raw/wine.csv`  
**Salidas:** `data/processed/train.csv`, `data/processed/test.csv` y `artifacts/data_contract.json`

> El conjunto de test se crea aquí, pero no debe explorarse ni utilizarse después del split.

## Objetivos

- cargar y validar datos externos;
- identificar problemas básicos de calidad;
- crear un split reproducible y estratificado;
- realizar EDA únicamente sobre entrenamiento;
- guardar datos y metadatos para la siguiente etapa.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 4)

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

### (Opcional) Generar wine.csv si no existe

Si no tienes `data/raw/wine.csv`, esta celda lo genera a partir del dataset Wine de scikit-learn.

In [ ]:
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH_CHECK = RAW_DATA_DIR / "wine.csv"

if not DATA_PATH_CHECK.exists():
    from sklearn.datasets import load_wine
    wine = load_wine(as_frame=True)
    df_gen = wine.frame.copy()
    # La columna target ya se llama 'target'
    df_gen.to_csv(DATA_PATH_CHECK, index=False)
    print("wine.csv generado desde sklearn.")
else:
    print("wine.csv ya existe.")

## 1. Carga y profiling

In [ ]:
DATA_PATH = RAW_DATA_DIR / "wine.csv"

# Carga el archivo en un DataFrame llamado df
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
# Profiling: forma, tipos, nulos, duplicados y distribución de target
print("Forma del dataset:", df.shape)
print("\n--- Tipos de datos ---")
print(df.dtypes)

print("\n--- Valores nulos por columna ---")
print(df.isnull().sum())

print("\n--- Filas duplicadas ---")
print(df.duplicated().sum())

print("\n--- Distribución de la clase (target) ---")
print(df["target"].value_counts())
print("\nProporciones:")
print(df["target"].value_counts(normalize=True))

print("\n--- Estadísticas descriptivas ---")
df.describe()

**Conclusión de calidad:**

> El dataset Wine tiene 178 filas y 14 columnas (13 features + target). No presenta valores nulos ni duplicados. La variable `target` tiene 3 clases con una distribución ligeramente desbalanceada (clase 1 es la más frecuente). Por eso se usará un split **estratificado** para preservar las proporciones. Las features están en escalas muy distintas (ej. `proline` en cientos vs `hue` cerca de 1), lo que indica que el escalado será importante para modelos sensibles a la escala.

## 2. Split train/test

In [ ]:
# Crea train_df (80 %) y test_df (20 %) con split estratificado
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["target"],
    random_state=RANDOM_STATE,
)

In [ ]:
# Verificación del split
assert len(train_df) + len(test_df) == len(df)
assert set(train_df.index).isdisjoint(set(test_df.index))
print("Train:", train_df.shape)
print("Test:", test_df.shape)

## 3. EDA sobre entrenamiento

In [ ]:
# Visualización 1: Distribución de clases
fig, ax = plt.subplots(figsize=(6, 4))
train_df["target"].value_counts().sort_index().plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Distribución de clases (train)")
ax.set_xlabel("Clase")
ax.set_ylabel("Conteo")
plt.tight_layout()
plt.show()

In [ ]:
# Visualización 2: Boxplots de algunas variables por clase
feature_cols = [c for c in train_df.columns if c != "target"]
key_features = feature_cols[:4]

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, feat in zip(axes.ravel(), key_features):
    sns.boxplot(data=train_df, x="target", y=feat, ax=ax)
    ax.set_title(f"{feat} por clase")
plt.tight_layout()
plt.show()

In [ ]:
# Visualización 3: Matriz de correlación
fig, ax = plt.subplots(figsize=(11, 9))
corr = train_df[feature_cols].corr()
sns.heatmap(corr, cmap="coolwarm", center=0, ax=ax, annot=False)
ax.set_title("Matriz de correlación de features (train)")
plt.tight_layout()
plt.show()

**Hallazgos:**

1. Las clases muestran diferencias notables en varias features (por ejemplo `flavanoids` y `proline` separan bien las clases), lo que sugiere que un clasificador debería alcanzar buena precisión.
2. Existen features altamente correlacionadas entre sí (como `total_phenols` y `flavanoids`), lo que podría introducir redundancia; modelos lineales podrían beneficiarse de regularización.
3. Las escalas de las variables son muy diferentes (ej. `proline` ~ cientos/miles vs `hue` ~ 0–2), por lo que es imprescindible estandarizar antes de usar modelos basados en distancia (KNN) o lineales.

## 4. Persistencia de datos y contrato

In [ ]:
# Reiniciar índices y guardar train.csv y test.csv sin índice
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

train_df.to_csv(PROCESSED_DATA_DIR / "train.csv", index=False)
test_df.to_csv(PROCESSED_DATA_DIR / "test.csv", index=False)

# Crear data_contract.json
features = [c for c in df.columns if c != "target"]
data_contract = {
    "target": "target",
    "features": features,
    "random_state": RANDOM_STATE,
    "test_size": 0.20,
}

(ARTIFACTS_DIR / "data_contract.json").write_text(
    json.dumps(data_contract, indent=2),
    encoding="utf-8",
)

print("Contrato de datos:")
print(json.dumps(data_contract, indent=2))

In [ ]:
# Verificación de entrega
assert (PROCESSED_DATA_DIR / "train.csv").exists()
assert (PROCESSED_DATA_DIR / "test.csv").exists()
assert (ARTIFACTS_DIR / "data_contract.json").exists()
print("Challenge 1 completado.")

## Checklist

- [x] Cargué el CSV desde `data/raw`.
- [x] Revisé calidad y distribución de la clase.
- [x] Realicé un split estratificado.
- [x] No exploré el conjunto de test.
- [x] Guardé los tres artefactos requeridos.